# Михаил, 3 версия нейросети анализа изображений Oskelly

## Setup

In [1]:
! pip install comet_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.6 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.4.1
    Uninstalling python-box-7.4.1:
      Successfully uninstalled python-box-7.4.1


In [2]:
import comet_ml

In [4]:
! pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00


In [3]:
# Пришлось перенести все импорты вверх, чтобы везде зафиксировать сид
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [5]:
SEED = 12062026


np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


## Model (part1)


Возьмем ViT OpenAi, это наш энкодер. Нам нужно это для преобразовани картинок в датасете по правилам ViT, поэтому начнем сейчас

In [6]:
import open_clip

In [7]:
args = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = args[0]
preprocesser = args[1]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## Data

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from pathlib import Path
p = '/content/drive/MyDrive/GP5_final/oskelly_dataset.xlsx'
df = pd.read_excel(p)

In [10]:
df.shape

(9600, 23)

In [11]:
sl = {
    'Вечерние платья': 'Платья',
    'Другие платья': 'Платья',
    'Коктейльные платья': 'Платья',
    'Повседневные платья': 'Платья',
    'Сарафаны': 'Платья',
    'Юбки макси': 'Юбки',
    'Юбки миди': 'Юбки',
    'Юбки мини': 'Юбки',
    'Юбки-шорты': 'Юбки',
    'Брюки узкие': 'Брюки',
    'Брюки широкие': 'Брюки',
    'Прямые брюки': 'Брюки',
    'Кюлоты': 'Брюки',
    'Бриджи': 'Брюки',
    'Леггинсы и велосипедки': 'Брюки',
    'Зауженные джинсы': 'Джинсы',
    'Прямые джинсы': 'Джинсы',
    'Расклешенные джинсы': 'Джинсы',
    'Шорты': 'Шорты',
    'Блузы': 'Блузы и рубашки',
    'Рубашки': 'Блузы и рубашки',
    'Водолазки': 'Топы с длинным рукавом',
    'Лонгсливы': 'Топы с длинным рукавом',
    'Майки': 'Майки и футболки',
    'Футболки': 'Майки и футболки',
    'Туники': 'Майки и футболки',
    'Джемперы и свитеры': 'Трикотаж',
    'Кардиганы': 'Трикотаж',
    'Худи и толстовки': 'Худи и толстовки',
    'Пальто': 'Верхняя одежда',
    'Куртки': 'Верхняя одежда',
    'Пуховики': 'Верхняя одежда',
    'Парки': 'Верхняя одежда',
    'Тренчи и плащи': 'Верхняя одежда',
    'Дубленки': 'Верхняя одежда',
    'Шубы': 'Верхняя одежда',
    'Жакеты и пиджаки': 'Жакеты и пиджаки',
    'Накидки и пончо': 'Верхняя одежда',
    'Жилетки': 'Жилеты',
    'Жилеты': 'Жилеты',
    'Костюмы с брюками': 'Костюмы и комплекты',
    'Костюмы с юбками': 'Костюмы и комплекты',
    'Комплекты': 'Костюмы и комплекты',
    'Комбинезоны': 'Комбинезоны',
    'Бюстгалтеры': 'Белье',
    'Корсеты': 'Белье',
    'Боди': 'Боди',
    'Носки, чулки и колготы': 'Носки, чулки и колготы',
    'Купальники': 'Купальники и пляж',
    'Парео': 'Купальники и пляж',
    'Спортивные брюки и шорты': 'Спортивная одежда',
    'Спортивные костюмы': 'Спортивная одежда',
    'Спортивные куртки': 'Спортивная одежда',
    'Пижамы': 'Пижамы',
    'Пижамы и сорочки': 'Пижамы',
}

In [12]:
df['Категория для label'] = df['Категория'].map(sl)

In [13]:
sub = df.groupby(by=['Категория для label'])['image_path'].count().reset_index().sort_values(by=['image_path'])
top_labels = sub[sub['image_path'] > len(df)//100]['Категория для label'].values
top_labels

array(['Жилеты', 'Жакеты и пиджаки', 'Топы с длинным рукавом', 'Юбки',
       'Брюки', 'Шорты', 'Блузы и рубашки', 'Джинсы', 'Трикотаж',
       'Худи и толстовки', 'Платья', 'Верхняя одежда', 'Майки и футболки'],
      dtype=object)

In [14]:
sub

,Категория для label,image_path
12,"Носки, чулки и колготы",3
2,Боди,12
0,Белье,25
8,Комбинезоны,26
9,Костюмы и комплекты,60
10,Купальники и пляж,78
14,Спортивная одежда,87
7,Жилеты,120
6,Жакеты и пиджаки,197
15,Топы с длинным рукавом,211


In [14]:
df = df[df['Категория для label'].isin(top_labels)]
df = df.dropna(subset=['Категория для label', 'image_path']).reset_index(drop=True)
df.shape

(7717, 24)

In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['Категория для label'])
lst_names = list(le.classes_)

In [16]:
len(lst_names)

13

In [17]:
y = torch.tensor(df['label'].values)

In [17]:
from tqdm import tqdm

In [18]:
from PIL import Image
path = '/content/drive/MyDrive/GP5_final/oskelly_images/'
df['image_path_ok'] = df['image_path'].apply(lambda x: path + str(x).split('oskelly_images')[-1][1:])

raw_images = []
ps = list(df['image_path_ok'].values)
for img_p in tqdm(ps):
  inp = preprocesser(Image.open(img_p).convert('RGB'))
  raw_images.append(inp)


100%|██████████| 7717/7717 [44:46<00:00,  2.87it/s]


In [19]:
raw_imagesS = torch.stack(raw_images, dim=0)

In [ ]:
# save
torch.save(raw_imagesS, '/content/drive/MyDrive/GP5_final/raw_imagesS.pt')

In [19]:
# unsave
raw_imagesS = torch.load('/content/drive/MyDrive/GP5_final/raw_imagesS.pt')

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [19]:
clip_model.to(device)
clip_model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [22]:
X = []
batch_size = 16
for i in tqdm(range(0, len(raw_imagesS), batch_size)):
  batch = raw_imagesS[i:i+batch_size].to(device)
  with torch.no_grad():
    emb = clip_model.encode_image(batch)
  raw_imagesS[i:i+batch_size] = 0
  X.append(emb.cpu())
X = torch.cat(X, dim=0)

X.shape

100%|██████████| 483/483 [00:21<00:00, 22.38it/s]


torch.Size([7717, 512])

In [23]:
# save
torch.save(X, '/content/drive/MyDrive/GP5_final/clip_embeddings_X_1406.pt')


In [ ]:
torch.save(y, '/content/drive/MyDrive/GP5_final/labels_y_1406.pt')

In [24]:
del raw_imagesS

In [20]:
# unsave
X = torch.load('/content/drive/MyDrive/GP5_final/clip_embeddings_X_1406.pt')
y = torch.load('/content/drive/MyDrive/GP5_final/labels_y_1406.pt')

In [21]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [22]:
train_loadet = DataLoader(TensorDataset(X_train,  y_train),  batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

## Model (part2)


In [23]:
for param in clip_model.parameters():
  param.requires_grad_(False)

In [24]:
# Бейзлайн модель, простая MLP на выходах VIT
class ModelX(nn.Module):
  def __init__(self, ViT, dropout=0.3):
    super().__init__()
    self.ViT = ViT
    self.fc1 = nn.Linear(512, 256)
    self.fc2 = nn.Linear(256, 64)
    self.fc3 = nn.Linear(64, 13)
    self.norm1 = nn.LayerNorm(256)
    self.norm2 = nn.LayerNorm(64)
    self.act1 = nn.ReLU()
    self.act2 = nn.ReLU()
    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)


  def forward(self, cache_embed):
    x = self.fc1(cache_embed)
    x = self.act1(self.norm1(x))
    x = self.drop1(x)
    x = self.fc2(x)
    x = self.act2(self.norm2(x))
    x = self.drop2(x)
    logits = self.fc3(x)
    return logits

  def predict(self, x):
    cache_embed = self.ViT.encode_image(x).float()
    x = self.fc1(cache_embed)
    x = self.act1(self.norm1(x))
    x = self.drop1(x)
    x = self.fc2(x)
    x = self.act2(self.norm2(x))
    x = self.drop2(x)
    logits = self.fc3(x)
    probs = logits.softmax(dim=1)
    return probs

# Train + comet logging



In [ ]:
# Дали рекомендацию коллеги с ФКН, т.к. не работал WB
# https://www.comet.com/docs/opik/tracing/advanced/log_traces
# APIKEY : wX0NkEZpsfuU3t3cmf15A6HzP

In [25]:
from comet_ml import Experiment, Artifact

In [28]:

experiment0 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
art = Artifact("clip_embeddings", artifact_type="dataset",
               metadata={"encoder": "ViT-B-32/openai", "n": int(len(y)),
                         "n_classes": int(len(lst_names)), "seed": SEED})
art.add("/content/drive/MyDrive/GP5_final/clip_embeddings_X_1406.pt")
art.add("/content/drive/MyDrive/GP5_final/labels_y_1406.pt")
experiment0.log_artifact(art)
experiment0.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/aedf9f2c17be49e5a34be7c49598219d

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
COMET INFO: Artifact 'clip_embeddings' version 8.0.0 created (previous was: 7.0.0)
COMET INFO: Scheduling the upload of 2 assets: 2 local assets for a size of 15.13 MB, and 0 remote assets (will be linked, not uploaded). This can take some time.
COMET INFO: Artifact 'gp5-team1/clip_embeddings:8.0.0' has started uploading asynchronously
COMET INFO: Artifact 'gp5-team1/clip_embeddings:8.0.0' has been fully uploaded successfully
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET IN

In [26]:
best_f1 = 0

In [27]:
from sklearn.metrics import accuracy_score, f1_score

def train(NUM_EPOCHS, LR, model, experiment, crit=nn.CrossEntropyLoss()):
  global best_f1
  model = model.to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=LR)
  criterion = crit
  experiment.log_parameters({
            "epochs": NUM_EPOCHS,
            "lr": LR,
            "batch_size": 128,
            "model": 'CLIPHEAD'
        })


  for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0
    for embs, labels in train_loadet:
      embs, labels = embs.to(device), labels.to(device)
      optimizer.zero_grad()
      loss = criterion(model(embs), labels)
      loss.backward()
      optimizer.step()
      train_loss += loss.item()
    train_loss /= len(train_loadet)

    # оценка в разрезе эпох из критериев....
    model.eval()
    test_loss = 0
    all_preds = []
    all_true = []
    with torch.no_grad():
      for embs, labels in test_loader:
        embs, labels = embs.to(device), labels.to(device)
        logits = model(embs)
        test_loss += criterion(logits, labels).item()
        all_preds += logits.argmax(dim=1).tolist()
        all_true += labels.tolist()
    test_loss /= len(test_loader)
    test_acc = accuracy_score(all_true, all_preds)
    test_f1  = f1_score(all_true, all_preds, average='macro')
    experiment.log_metrics({
              "train_loss": train_loss,
                "test_loss": test_loss,
                "test_acc": test_acc,
                "test_macro_f1": test_f1,
            }, epoch=epoch)


    if test_f1 > best_f1:
      best_f1 = test_f1
      torch.save(model.state_dict(), '/content/drive/MyDrive/GP5_final/best_CLIPHEAD.pt')
      experiment.log_model('best_CLIPHEAD', '/content/drive/MyDrive/GP5_final/best_CLIPHEAD.pt')
  experiment.end()
  return model

In [28]:
NUM_EPOCHS = 100
LR = 1e-4
base_model = ModelX(clip_model, dropout=0.25)

experiment1 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
experiment1.set_name("CLIPHEAD BASE")
experiment1.add_tags(["CLIPHEAD", "dropout-0.25"])

train(NUM_EPOCHS, LR, base_model, experiment1)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/6f9b677f96ac4d7c92ddd7b043cb805a

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 100/100 [07:55<00:00,  4.76s/it]
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CLIPHEAD BASE
COMET INFO:     url             

ModelX(
  (ViT): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )
        )
      )
 

## Experiments (будет доработано)ёё


### Разные dropout

In [29]:
NUM_EPOCHS = 100
LR = 1e-4
model_dr0 = ModelX(clip_model, dropout=0)

experiment2 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
experiment2.set_name("CLIPHEAD base_d00")
experiment2.add_tags(["CLIPHEAD", "dropout-0"])

train(NUM_EPOCHS, LR, model_dr0, experiment2)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/96f28fe435c744e6bfce5fb3842fce19

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 100/100 [00:21<00:00,  4.60it/s]
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CLIPHEAD base_d00
COMET INFO:     url         

ModelX(
  (ViT): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )
        )
      )
 

In [30]:
NUM_EPOCHS = 100
LR = 1e-4
model_dr050 = ModelX(clip_model, dropout=0.5)

experiment2 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
experiment2.set_name("CLIPHEAD base_d05")
experiment2.add_tags(["CLIPHEAD", "dropout-0.5"])

train(NUM_EPOCHS, LR, model_dr050, experiment2)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/180ad8d4b9d145f6a0ceaf25ed9112c9

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 100/100 [00:56<00:00,  1.78it/s]
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CLIPHEAD base_d05
COMET INFO:     url         

ModelX(
  (ViT): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )
        )
      )
 

### Короткий MLP на выходе

In [31]:
# Бейзлайн модель, простая MLP на выходах VIT
class ModelY(nn.Module):
  def __init__(self, ViT, dropout=0.3):
    super().__init__()
    self.ViT = ViT
    self.fc1 = nn.Linear(512, 256)
    self.fc2 = nn.Linear(256, 13)
    self.norm1 = nn.LayerNorm(256)
    self.act1 = nn.ReLU()
    self.drop1 = nn.Dropout(dropout)


  def forward(self, cache_embed):
    x = self.fc1(cache_embed)
    x = self.act1(self.norm1(x))
    x = self.drop1(x)
    logits = self.fc2(x)
    return logits

  def predict(self, x):
    pass

In [32]:
NUM_EPOCHS = 100
LR = 1e-4
model_short = ModelY(clip_model, dropout=0.25)

experiment3 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
experiment3.set_name("CLIPHEAD small")
experiment3.add_tags(["CLIPHEAD", "small"])

train(NUM_EPOCHS, LR, model_short, experiment3)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/299f073edc5e4ec08d3515346cec408c

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 100/100 [00:19<00:00,  5.06it/s]
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CLIPHEAD small
COMET INFO:     url            

ModelY(
  (ViT): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )
        )
      )
 

In [35]:
from sklearn.metrics import classification_report
model_short.eval()
criterion = nn.CrossEntropyLoss()
if True:
  test_loss = 0
  all_preds = []
  all_true = []
  with torch.no_grad():
    for embs, labels in test_loader:
      embs, labels = embs.to(device), labels.to(device)
      logits = model_short(embs)
      test_loss += criterion(logits, labels).item()
      all_preds += logits.argmax(dim=1).tolist()
      all_true += labels.tolist()
  test_loss /= len(test_loader)

# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from sklearn.metrics import classification_report
print(classification_report(
    all_true, all_preds,
    labels=list(range(len(lst_names))),
    target_names=lst_names,
    zero_division=0))

                        precision    recall  f1-score   support

       Блузы и рубашки       0.67      0.69      0.68        80
                 Брюки       0.91      0.88      0.90        73
        Верхняя одежда       0.83      0.89      0.86       206
                Джинсы       0.92      0.94      0.93        98
      Жакеты и пиджаки       0.79      0.68      0.73        40
                Жилеты       0.88      0.62      0.73        24
      Майки и футболки       0.89      0.96      0.93       369
                Платья       0.90      0.91      0.91       164
Топы с длинным рукавом       0.72      0.43      0.54        42
              Трикотаж       0.80      0.68      0.73       152
      Худи и толстовки       0.83      0.86      0.84       160
                 Шорты       0.92      0.96      0.94        76
                  Юбки       0.91      0.88      0.90        60

              accuracy                           0.86      1544
             macro avg       0.85     

### Более глубокий MLP на выходе

In [33]:
# Бейзлайн модель, простая MLP на выходах VIT
class ModelZ(nn.Module):
  def __init__(self, ViT, dropout=0.3):
    super().__init__()
    self.ViT = ViT
    self.fc1 = nn.Linear(512, 256)
    self.fc2 = nn.Linear(256, 128)
    self.fc3 = nn.Linear(128, 64)
    self.fc4 = nn.Linear(64, 13)
    self.norm1 = nn.LayerNorm(256)
    self.norm2 = nn.LayerNorm(128)
    self.norm3 = nn.LayerNorm(64)
    self.act1 = nn.ReLU()
    self.act2 = nn.ReLU()
    self.act3 = nn.ReLU()
    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)
    self.drop3 = nn.Dropout(dropout)


  def forward(self, cache_embed):
    x = self.fc1(cache_embed)
    x = self.act1(self.norm1(x))
    x = self.drop1(x)
    x = self.fc2(x)
    x = self.act2(self.norm2(x))
    x = self.drop2(x)
    x = self.fc3(x)
    x = self.act3(self.norm3(x))
    x = self.drop3(x)
    logits = self.fc4(x)
    return logits

  def predict(self, x):
    pass

In [34]:
NUM_EPOCHS = 100
LR = 1e-4
model_big = ModelZ(clip_model, dropout=0.25)

experiment4 = Experiment(api_key='wX0NkEZpsfuU3t3cmf15A6HzP', project_name="oskelly-cv", workspace="gp5_team1")
experiment4.set_name("CLIPHEAD big")
experiment4.add_tags(["CLIPHEAD", "big"])

train(NUM_EPOCHS, LR, model_big, experiment4)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-cv/e7758e482b5b4411a39d6d1b9c0e3a96

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 100/100 [00:24<00:00,  4.03it/s]
COMET WARNING: Couldn't retrieve and log Google Colab notebook content, reason: 'NoneType' object is not subscriptable
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CLIPHEAD big
COMET INFO:     url              

ModelZ(
  (ViT): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Identity()
          )
        )
      )
 

In [38]:
from sklearn.metrics import classification_report
model_big.eval()
criterion = nn.CrossEntropyLoss()
if True:
  test_loss = 0
  all_preds = []
  all_true = []
  with torch.no_grad():
    for embs, labels in test_loader:
      embs, labels = embs.to(device), labels.to(device)
      logits = model_big(embs)
      test_loss += criterion(logits, labels).item()
      all_preds += logits.argmax(dim=1).tolist()
      all_true += labels.tolist()
  test_loss /= len(test_loader)

# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from sklearn.metrics import classification_report
print(classification_report(
    all_true, all_preds,
    labels=list(range(len(lst_names))),
    target_names=lst_names,
    zero_division=0))

                        precision    recall  f1-score   support

       Блузы и рубашки       0.66      0.70      0.68        80
                 Брюки       0.85      0.92      0.88        73
        Верхняя одежда       0.81      0.90      0.85       206
                Джинсы       0.94      0.91      0.92        98
      Жакеты и пиджаки       0.76      0.65      0.70        40
                Жилеты       0.89      0.67      0.76        24
      Майки и футболки       0.91      0.95      0.93       369
                Платья       0.89      0.91      0.90       164
Топы с длинным рукавом       0.60      0.36      0.45        42
              Трикотаж       0.74      0.70      0.72       152
      Худи и толстовки       0.84      0.80      0.82       160
                 Шорты       0.93      0.92      0.93        76
                  Юбки       0.89      0.83      0.86        60

              accuracy                           0.85      1544
             macro avg       0.82     